[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/22.%20The%20Internal%20Vehicle%20%28Terminal%20Truck%29%20Dispatching%20Problem/P22-Tier-7_executed.ipynb)

---


# 22. The Internal Vehicle (Terminal Truck) Dispatching Problem
## Tier 7: Human–AI Symbiotic Partnership

### Goal
This notebook demonstrates an explainable, human-in-the-loop truck-dispatching prototype. A rule-based AI component produces a recommendation and a human dispatcher can accept, modify, or reject it. The system records feedback and updates a simple trust score. The example focuses on how recommendations can be presented and discussed; it does not train a predictive model.

### Inputs and outputs
The inputs are container priorities, origin coordinates, deadlines, truck locations, availability times, fuel levels, maintenance hours, and reliability scores. The outputs are per-container truck recommendations, a record of the reasoning and feedback, and illustrative interaction metrics.

### Scope and model assumptions
- Coordinates use a rectilinear grid, so travel distance is Manhattan distance: $d_{ij}=|x_j-x_i|+|y_j-y_i|$. Coordinates are in distance units and speed is in distance units per minute.
- All times are relative to the start of the demonstration and measured in minutes. A deadline is the latest permitted completion time.
- Each container is evaluated independently. This is a local dispatch heuristic, not a globally coordinated schedule: it does not reserve trucks, model travel from one completed job to the next, or guarantee that multiple recommendations can be executed simultaneously.
- The score weights and confidence rules are illustrative design choices, not estimates learned from operational data. The notebook contains no measured accuracy, training-time, or flexibility evaluation.
- Human feedback is recorded and used in a transparent trust update. It does not retrain the dispatch rule or establish that future recommendations become more accurate.
- Historical records are not provided. Therefore, no historical-case evidence is presented.

### Dispatch rule
For container $i$ and truck $j$, let $p_i$ be container priority, $t_{ij}=d_{ij}/v$ be travel time, $a_j$ be truck availability time, $T_i$ be the deadline, and $t_0$ be the current time. A truck is feasible when

$$\max(a_j,t_0)+t_{ij}\leq T_i.$$

Among feasible trucks, the heuristic selects the largest score

$$U_{ij}=10p_i-2t_{ij}-20\,\mathbf{1}(f_j<30)-15\,\mathbf{1}(m_j>40),$$

where $f_j$ is fuel level, $m_j$ is accumulated maintenance hours, and $\mathbf{1}(\cdot)$ equals 1 when its condition is true and 0 otherwise. If no truck is feasible, the container receives no assignment. The code applies this rule separately to each container; it does not solve a joint assignment optimization problem.

### Learning objectives
By the end, readers should be able to distinguish a recommendation from an optimized schedule, trace a score to its inputs, interpret heuristic confidence cautiously, and explain what feedback does—and does not—change in this prototype.

In [ ]:
# Import the libraries used for data structures, the dispatch demonstration, and plots.
import copy
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np

# Keep the seeded random generators available for reproducible extensions.
np.random.seed(42)
random.seed(42)

print("Initializing the human–AI dispatch demonstration.")

In [ ]:
class ExplanationType(Enum):
    DECISION_TREE = "decision_tree"
    COUNTERFACTUAL = "counterfactual"
    FEATURE_IMPORTANCE = "feature_importance"
    CONFIDENCE_METRICS = "confidence_metrics"
    HISTORICAL_COMPARISON = "historical_comparison"


@dataclass
class AIRecommendation:
    """A dispatch recommendation and the information used to explain it."""

    recommendation_id: str
    task_type: str
    recommended_action: Dict[str, Any]
    confidence_score: float
    reasoning_path: List[Dict[str, Any]]
    feature_importance: Dict[str, float]
    counterfactuals: List[Dict[str, Any]]
    historical_cases: List[Dict[str, Any]]
    timestamp: float
    human_feedback: Optional[Dict[str, Any]] = None


@dataclass
class HumanDispatcher:
    """Dispatcher profile used to identify an interaction participant."""

    dispatcher_id: str
    experience_years: int
    expertise_areas: List[str]
    decision_history: List[Dict[str, Any]]
    preferences: Dict[str, float]
    trust_level: float = 0.5
    learning_rate: float = 0.1


@dataclass
class InteractionSession:
    """Record of recommendations, decisions, and feedback in a session."""

    session_id: str
    dispatcher_id: str
    ai_recommendations: List[AIRecommendation]
    human_decisions: List[Dict[str, Any]]
    feedback_provided: List[Dict[str, Any]]
    learning_outcomes: Dict[str, float]
    session_duration: float
    timestamp: float


@dataclass
class ExplainableComponent:
    """Container for a typed explanation and its confidence value."""

    component_type: ExplanationType
    explanation_data: Dict[str, Any]
    confidence: float
    visual_elements: List[Dict[str, Any]]


print("Interaction data structures are ready.")

In [ ]:
class FeedbackProcessor:
    """Count feedback actions without claiming that the dispatch rule is retrained."""

    def __init__(self):
        self.feedback_history = []
        self.learning_patterns = defaultdict(
            lambda: {"acceptances": 0, "rejections": 0, "modifications": 0}
        )

    def process_feedback(self, feedback: Dict[str, Any], recommendation: AIRecommendation):
        action = feedback.get("action")
        decision_type = feedback.get("decision_type", recommendation.task_type)
        self.feedback_history.append(
            {
                "timestamp": time.time(),
                "action": action,
                "decision_type": decision_type,
                "recommendation_id": recommendation.recommendation_id,
            }
        )
        action_to_count = {
            "accept": "acceptances",
            "reject": "rejections",
            "modify": "modifications",
        }
        if action in action_to_count:
            self.learning_patterns[decision_type][action_to_count[action]] += 1


class ConfidenceEstimator:
    """Produce a bounded heuristic indicator, not a calibrated probability."""

    def __init__(self):
        self.confidence_history = []

    def estimate_confidence(self, context: Dict[str, Any], recommendation: Dict[str, Any]) -> float:
        confidence = 0.8
        containers = context.get("containers", [])
        trucks = context.get("trucks", [])

        if len(containers) > 10 or len(trucks) > 15:
            confidence -= 0.1
        if any(container.get("priority", 0) >= 15 for container in containers):
            confidence += 0.05
        available_trucks = sum(
            truck.get("available_time", 0) <= context.get("current_time", 0)
            for truck in trucks
        )
        if available_trucks < len(containers):
            confidence -= 0.15

        confidence = max(0.1, min(0.95, confidence))
        self.confidence_history.append(confidence)
        return confidence


class ExplanationGenerator:
    """Format decision traces and heuristic factor weights for readers."""

    def generate_explanation(self, explanation_type: ExplanationType, data: Dict[str, Any]) -> str:
        if explanation_type == ExplanationType.DECISION_TREE:
            return self._decision_tree_template(data)
        if explanation_type == ExplanationType.COUNTERFACTUAL:
            return self._counterfactual_template(data)
        if explanation_type == ExplanationType.FEATURE_IMPORTANCE:
            return self._feature_importance_template(data)
        return "No explanation is available for this type."

    @staticmethod
    def _decision_tree_template(data: Dict[str, Any]) -> str:
        lines = ["Recommendation reasoning:"]
        for step in data.get("reasoning_path", []):
            lines.append(
                f"Step {step['step']}: {step['description']} — {step['decision']} "
                f"(trace indicator: {step['confidence']:.0%})"
            )
        return "\n".join(lines)

    @staticmethod
    def _counterfactual_template(data: Dict[str, Any]) -> str:
        scenarios = data.get("counterfactuals", [])
        if not scenarios:
            return "No counterfactual scenarios were computed."
        return "What-if scenarios:\n" + "\n".join(
            f"- {item['scenario']}: {item['outcome']}" for item in scenarios
        )

    @staticmethod
    def _feature_importance_template(data: Dict[str, Any]) -> str:
        weights = data.get("feature_importance", {})
        return "Heuristic score-factor weights (not learned feature importance):\n" + "\n".join(
            f"- {name.replace('_', ' ').title()}: {weight:.1%}"
            for name, weight in sorted(weights.items(), key=lambda item: item[1], reverse=True)
        )


class ConfidenceCalibrator:
    """Compare a heuristic indicator with dispatcher-provided feedback."""

    def __init__(self):
        self.calibration_history = []

    def update(self, recommendation: AIRecommendation, confidence_feedback: float):
        if not 0.0 <= confidence_feedback <= 1.0:
            raise ValueError("Confidence feedback must be between 0 and 1.")
        self.calibration_history.append(
            {
                "recommendation_id": recommendation.recommendation_id,
                "ai_confidence": recommendation.confidence_score,
                "human_feedback": confidence_feedback,
            }
        )

    def get_calibration_score(self) -> float:
        if not self.calibration_history:
            return 0.5
        recent = self.calibration_history[-10:]
        mean_absolute_error = np.mean(
            [abs(item["ai_confidence"] - item["human_feedback"]) for item in recent]
        )
        return float(1.0 - mean_absolute_error)


class TrustModel:
    """Maintain a bounded, explicitly illustrative dispatcher trust score."""

    def __init__(self):
        self.trust_scores = {}
        self.trust_history = defaultdict(list)

    def calculate_trust(self, recommendation: AIRecommendation, session: InteractionSession) -> float:
        base = self.trust_scores.get(session.dispatcher_id, 0.5)
        confidence_adjustment = (recommendation.confidence_score - 0.5) * 0.2
        reasoning_quality = min(len(recommendation.reasoning_path) / 5.0, 1.0)
        reasoning_adjustment = (reasoning_quality - 0.5) * 0.1
        return max(0.0, min(1.0, base + confidence_adjustment + reasoning_adjustment))

    def update_trust(self, dispatcher_id: str, decision: Dict[str, Any]):
        action = decision.get("action")
        adjustment = {"accept": 0.05, "modify": 0.02, "reject": -0.03}.get(action, 0.0)
        updated = max(0.0, min(1.0, self.trust_scores.get(dispatcher_id, 0.5) + adjustment))
        self.trust_scores[dispatcher_id] = updated
        self.trust_history[dispatcher_id].append(updated)
        return updated

    def get_current_trust(self, dispatcher_id: str) -> float:
        return self.trust_scores.get(dispatcher_id, 0.5)


print("Feedback, confidence, explanation, and trust components are ready.")

In [ ]:
class ExplainableAISystem:
    """Apply the stated per-container dispatch heuristic and expose its reasoning."""

    def __init__(self, system_id: str):
        self.system_id = system_id
        self.confidence_estimator = ConfidenceEstimator()
        self.explanation_generator = ExplanationGenerator()
        self.feedback_processor = FeedbackProcessor()
        self.confidence_calibrators = ConfidenceCalibrator()
        self.recommendation_history = []

    def generate_recommendation(
        self, context: Dict[str, Any], dispatcher_profile: HumanDispatcher
    ) -> AIRecommendation:
        base = self._analyze_context(context)
        confidence = self.confidence_estimator.estimate_confidence(context, base)
        reasoning = self._generate_reasoning_path(context, base)
        features = self._calculate_feature_importance(context, base)
        counterfactuals = self._generate_counterfactuals(context, base)

        recommendation = AIRecommendation(
            recommendation_id=f"rec_{len(self.recommendation_history) + 1:04d}",
            task_type=base["task_type"],
            recommended_action=base["action"],
            confidence_score=confidence,
            reasoning_path=reasoning,
            feature_importance=features,
            counterfactuals=counterfactuals,
            historical_cases=[],
            timestamp=time.time(),
        )
        self.recommendation_history.append(recommendation)
        return recommendation

    def _analyze_context(self, context: Dict[str, Any]) -> Dict[str, Any]:
        containers = context.get("containers", [])
        trucks = context.get("trucks", [])
        current_time = context.get("current_time", 0.0)
        assignments = []

        for container in containers:
            best = self._find_best_truck(container, trucks, current_time)
            if best is not None:
                assignments.append(
                    {
                        "container_id": container["id"],
                        "truck_id": best["id"],
                        "reason": best["reason"],
                        "travel_time": best["travel_time"],
                        "estimated_completion": best["completion_time"],
                        "deadline": container["deadline"],
                        "priority": container["priority"],
                        "utility": best["utility"],
                        "confidence": best["confidence"],
                    }
                )
        return {"task_type": "truck_dispatching", "action": {"assignments": assignments}}

    @staticmethod
    def _score_truck(container: Dict[str, Any], truck: Dict[str, Any], current_time: float):
        distance = abs(truck["location"][0] - container["origin"][0]) + abs(
            truck["location"][1] - container["origin"][1]
        )
        travel_time = distance / 25.0
        completion_time = max(truck["available_time"], current_time) + travel_time
        if completion_time > container["deadline"]:
            return None

        utility = container["priority"] * 10 - travel_time * 2
        if truck.get("fuel_level", 100) < 30:
            utility -= 20
        if truck.get("maintenance_hours", 0) > 40:
            utility -= 15
        reliability = truck.get("reliability_score", 0.5)
        confidence = min(0.9, 0.5 + reliability * 0.4)
        return {
            "id": truck["id"],
            "utility": utility,
            "travel_time": travel_time,
            "completion_time": completion_time,
            "confidence": confidence,
        }

    def _find_best_truck(
        self, container: Dict[str, Any], trucks: List[Dict[str, Any]], current_time: float
    ) -> Optional[Dict[str, Any]]:
        candidates = []
        for truck in trucks:
            candidate = self._score_truck(container, truck, current_time)
            if candidate is not None:
                candidate["reason"] = self._generate_truck_reason(
                    truck, container, candidate["travel_time"], candidate["utility"]
                )
                candidates.append(candidate)
        if not candidates:
            return None
        return max(candidates, key=lambda candidate: candidate["utility"])

    @staticmethod
    def _generate_truck_reason(
        truck: Dict[str, Any], container: Dict[str, Any], travel_time: float, utility: float
    ) -> str:
        return (
            f"Feasible for the deadline; travel time {travel_time:.2f} min; "
            f"priority {container['priority']}; utility {utility:.2f}; "
            f"fuel {truck.get('fuel_level', 100):.0f}; "
            f"maintenance hours {truck.get('maintenance_hours', 0):.0f}."
        )

    @staticmethod
    def _generate_reasoning_path(context: Dict[str, Any], recommendation: Dict[str, Any]):
        containers = context.get("containers", [])
        assignments = recommendation["action"]["assignments"]
        unassigned = len(containers) - len(assignments)
        return [
            {"step": 1, "description": "Read the dispatch request", "decision": f"{len(containers)} containers supplied", "confidence": 1.0},
            {"step": 2, "description": "Check truck feasibility", "decision": "Deadline feasibility is evaluated for each container–truck pair", "confidence": 0.95},
            {"step": 3, "description": "Score feasible trucks", "decision": "Priority, travel time, fuel, and maintenance enter the stated utility score", "confidence": 0.9},
            {"step": 4, "description": "Select per-container recommendations", "decision": f"{len(assignments)} assigned; {unassigned} have no feasible truck", "confidence": 0.85},
            {"step": 5, "description": "Interpret the scope", "decision": "Recommendations are independent and are not a joint schedule", "confidence": 1.0},
        ]

    @staticmethod
    def _calculate_feature_importance(context: Dict[str, Any], recommendation: Dict[str, Any]):
        # These normalized values describe the heuristic score design, not learned effects.
        weights = {
            "container_priority": 0.25,
            "travel_time": 0.20,
            "deadline_feasibility": 0.20,
            "truck_availability": 0.15,
            "fuel_level": 0.10,
            "maintenance_status": 0.10,
        }
        total = sum(weights.values())
        return {name: value / total for name, value in weights.items()}

    def _generate_counterfactuals(
        self, context: Dict[str, Any], recommendation: Dict[str, Any]
    ) -> List[Dict[str, Any]]:
        trucks = context.get("trucks", [])
        current_time = context.get("current_time", 0.0)
        scenarios = []
        for assignment in recommendation["action"]["assignments"][:3]:
            container = next(
                item for item in context["containers"] if item["id"] == assignment["container_id"]
            )
            alternatives = []
            for truck in trucks:
                if truck["id"] == assignment["truck_id"]:
                    continue
                candidate = self._score_truck(container, truck, current_time)
                if candidate is not None:
                    alternatives.append(candidate)
            if alternatives:
                alternative = max(alternatives, key=lambda item: item["utility"])
                scenarios.append(
                    {
                        "scenario": f"If {assignment['truck_id']} were unavailable for {container['id']}",
                        "outcome": f"The best other feasible truck by this score would be {alternative['id']} (utility {alternative['utility']:.2f}).",
                    }
                )
            else:
                scenarios.append(
                    {
                        "scenario": f"If {assignment['truck_id']} were unavailable for {container['id']}",
                        "outcome": "No other feasible truck is present in the supplied data.",
                    }
                )
        return scenarios

    def process_human_feedback(self, recommendation_id: str, feedback: Dict[str, Any]):
        recommendation = next(
            (item for item in self.recommendation_history if item.recommendation_id == recommendation_id),
            None,
        )
        if recommendation is None:
            raise ValueError(f"Unknown recommendation ID: {recommendation_id}")
        recommendation.human_feedback = feedback
        self.feedback_processor.process_feedback(feedback, recommendation)
        if "confidence_feedback" in feedback:
            self.confidence_calibrators.update(recommendation, feedback["confidence_feedback"])


print("The explainable per-container dispatch rule is ready.")

In [ ]:
class HumanAISymbioticInterface:
    """Present AI recommendations and record dispatcher decisions."""

    def __init__(self, ai_system: ExplainableAISystem):
        self.ai_system = ai_system
        self.active_sessions = {}
        self.interaction_history = []
        self.trust_model = TrustModel()
        self.collaboration_metrics = {
            "decisions_made": 0,
            "ai_acceptances": 0,
            "human_overrides": 0,
            "collaborative_improvements": 0,
        }

    def start_collaboration_session(self, dispatcher: HumanDispatcher, context: Dict[str, Any]) -> str:
        session_id = f"session_{len(self.active_sessions) + 1:04d}"
        recommendation = self.ai_system.generate_recommendation(context, dispatcher)
        self.active_sessions[session_id] = InteractionSession(
            session_id=session_id,
            dispatcher_id=dispatcher.dispatcher_id,
            ai_recommendations=[recommendation],
            human_decisions=[],
            feedback_provided=[],
            learning_outcomes={},
            session_duration=0.0,
            timestamp=time.time(),
        )
        return session_id

    def present_recommendation(self, session_id: str) -> Dict[str, Any]:
        session = self.active_sessions.get(session_id)
        if session is None:
            return {"error": "Session not found."}
        recommendation = session.ai_recommendations[0]
        generator = self.ai_system.explanation_generator
        explanation = {
            "summary": f"The heuristic recommends {len(recommendation.recommended_action['assignments'])} assignments.",
            "confidence": f"Heuristic confidence indicator: {recommendation.confidence_score:.1%}; this is not a calibrated probability.",
            "decision_tree": generator.generate_explanation(
                ExplanationType.DECISION_TREE, {"reasoning_path": recommendation.reasoning_path}
            ),
            "feature_importance": generator.generate_explanation(
                ExplanationType.FEATURE_IMPORTANCE, {"feature_importance": recommendation.feature_importance}
            ),
            "counterfactuals": generator.generate_explanation(
                ExplanationType.COUNTERFACTUAL, {"counterfactuals": recommendation.counterfactuals}
            ),
            "historical_cases": "No historical records were supplied.",
            "risk_assessment": self._generate_risk_assessment(recommendation),
        }
        return {
            "session_id": session_id,
            "recommendation": recommendation,
            "explanation": explanation,
            "trust_score": self.trust_model.calculate_trust(recommendation, session),
            "interaction_options": self._generate_interaction_options(),
        }

    @staticmethod
    def _generate_risk_assessment(recommendation: AIRecommendation) -> str:
        assignments = recommendation.recommended_action["assignments"]
        tight = [
            item for item in assignments
            if item["deadline"] - item["estimated_completion"] <= 5.0
        ]
        if tight:
            return f"Review recommended: {len(tight)} assignment(s) have at most 5 minutes of deadline slack."
        if not assignments:
            return "No assignments were produced; inspect deadlines and available resources."
        return "No assignment is within 5 minutes of its deadline in this demonstration."

    @staticmethod
    def _generate_interaction_options():
        return [
            {"action": "accept", "label": "Accept recommendation", "description": "Accept the proposed assignments."},
            {"action": "modify", "label": "Modify assignments", "description": "Record a human modification and its rationale."},
            {"action": "reject", "label": "Reject recommendation", "description": "Record a rejection and its rationale."},
            {"action": "explore", "label": "Explore explanation", "description": "Review the explanation without changing the recommendation."},
        ]

    def process_human_decision(self, session_id: str, decision: Dict[str, Any]) -> Dict[str, Any]:
        session = self.active_sessions.get(session_id)
        if session is None:
            return {"error": "Session not found."}
        action = decision.get("action")
        if action not in {"accept", "modify", "reject", "explore"}:
            raise ValueError("Action must be accept, modify, reject, or explore.")

        recommendation = session.ai_recommendations[0]
        session.human_decisions.append(decision)
        self.collaboration_metrics["decisions_made"] += 1
        if action == "accept":
            self.collaboration_metrics["ai_acceptances"] += 1
        elif action in {"modify", "reject"}:
            self.collaboration_metrics["human_overrides"] += 1

        feedback = {
            "action": action,
            "decision_type": recommendation.task_type,
            "reasoning": decision.get("reasoning", ""),
            "modifications": decision.get("modifications", []),
        }
        if "confidence_in_ai" in decision:
            feedback["confidence_feedback"] = decision["confidence_in_ai"]
        self.ai_system.process_human_feedback(recommendation.recommendation_id, feedback)
        session.feedback_provided.append(feedback)

        updated_trust = self.trust_model.update_trust(session.dispatcher_id, decision)
        outcomes = {
            "feedback_recorded": True,
            "dispatch_rule_retrained": False,
            "trust_change": updated_trust - self.trust_model.trust_history[session.dispatcher_id][-2]
            if len(self.trust_model.trust_history[session.dispatcher_id]) > 1
            else updated_trust - 0.5,
        }
        session.learning_outcomes.update(outcomes)
        self.interaction_history.append({"session_id": session_id, "action": action})
        return {
            "session_id": session_id,
            "decision_processed": True,
            "learning_outcomes": outcomes,
            "updated_trust": updated_trust,
            "feedback_acknowledged": True,
        }


print("The human–AI interaction interface is ready.")

In [ ]:
class AdaptationEngine:
    """Summarize feedback events without changing the dispatch heuristic."""

    def __init__(self):
        self.adaptation_history = []
        self.performance_metrics = defaultdict(list)

    def adapt_to_feedback(self, feedback: Dict[str, Any], recommendation: AIRecommendation):
        event = {
            "feedback_type": feedback.get("action"),
            "recommendation_id": recommendation.recommendation_id,
            "confidence_indicator": recommendation.confidence_score,
            "rule_changed": False,
        }
        self.adaptation_history.append(event)
        self.performance_metrics["feedback_processed"].append(1)
        self.performance_metrics["accepted"].append(int(feedback.get("action") == "accept"))
        return event


print("Feedback-event tracking is ready; no model retraining is performed.")

In [ ]:
print("\n" + "=" * 60)
print("HUMAN–AI DISPATCH SYSTEM INITIALIZATION")
print("=" * 60)

ai_system = ExplainableAISystem("terminal_dispatch_ai_demo")
symbiotic_interface = HumanAISymbioticInterface(ai_system)
adaptation_engine = AdaptationEngine()

human_dispatcher = HumanDispatcher(
    dispatcher_id="dispatcher_001",
    experience_years=12,
    expertise_areas=["truck dispatching", "priority management", "emergency response"],
    decision_history=[],
    preferences={"safety_first": 0.4, "efficiency": 0.3, "reliability": 0.3},
    trust_level=0.6,
)

print(f"AI system: {ai_system.system_id}")
print(f"Dispatcher: {human_dispatcher.dispatcher_id} ({human_dispatcher.experience_years} years of experience)")
print("The dispatch rule is illustrative; the profile does not alter its scores.")

# Times are minutes from the start of this example; coordinates are distance units.
urgent_container = {
    "id": "C-9876",
    "origin": (200, 100),
    "destination": (800, 300),
    "priority": 20,
    "deadline": 15.0,
    "contents": "critical medical supplies",
}
trucks = [
    {"id": "Truck-Alpha", "location": (180, 120), "available_time": 0.0, "fuel_level": 25.0, "maintenance_hours": 45.0, "reliability_score": 0.6},
    {"id": "Truck-Beta", "location": (250, 150), "available_time": 0.0, "fuel_level": 85.0, "maintenance_hours": 15.0, "reliability_score": 0.9},
    {"id": "Truck-Gamma", "location": (300, 200), "available_time": 5.0, "fuel_level": 60.0, "maintenance_hours": 25.0, "reliability_score": 0.8},
]
other_containers = [
    {"id": "C-5432", "origin": (100, 50), "destination": (600, 250), "priority": 8, "deadline": 45.0},
    {"id": "C-8765", "origin": (400, 100), "destination": (200, 350), "priority": 12, "deadline": 30.0},
]
operational_context = {
    "current_time": 0.0,
    "containers": [urgent_container] + other_containers,
    "trucks": trucks,
    "operational_status": "normal",
    "weather_conditions": "clear",
    "terminal_congestion": "moderate",
}

print(f"\nContainers: {len(operational_context['containers'])}")
print(f"Trucks: {len(operational_context['trucks'])}")
print(f"Urgent container deadline: {urgent_container['deadline']:.1f} minutes")
print("Truck-Alpha is closer to the urgent container but has low fuel and high maintenance hours.")
print("Truck-Beta is farther away but avoids those two score penalties.")

In [ ]:
print("\n" + "=" * 60)
print("RECOMMENDATION AND EXPLANATION")
print("=" * 60)

session_id = symbiotic_interface.start_collaboration_session(human_dispatcher, operational_context)
presentation = symbiotic_interface.present_recommendation(session_id)
if "error" in presentation:
    raise RuntimeError(presentation["error"])

recommendation = presentation["recommendation"]
assignments = recommendation.recommended_action["assignments"]
print(f"Session: {session_id}")
print(f"Heuristic confidence indicator: {recommendation.confidence_score:.1%}")
print(f"Assignments produced: {len(assignments)}")

urgent_assignment = next(
    (item for item in assignments if item["container_id"] == urgent_container["id"]), None
)
if urgent_assignment is not None:
    print("\nUrgent-container recommendation")
    print(f"Container: {urgent_assignment['container_id']}")
    print(f"Truck: {urgent_assignment['truck_id']}")
    print(f"Travel time: {urgent_assignment['travel_time']:.2f} minutes")
    print(f"Completion time: {urgent_assignment['estimated_completion']:.2f} minutes from start")
    print(f"Utility score: {urgent_assignment['utility']:.2f}")
    print(f"Reason: {urgent_assignment['reason']}")
else:
    print("No feasible truck was found for the urgent container.")

print("\n" + presentation["explanation"]["decision_tree"])
print("\n" + presentation["explanation"]["feature_importance"])
print("\n" + presentation["explanation"]["counterfactuals"])
print("\n" + presentation["explanation"]["risk_assessment"])
print(presentation["explanation"]["historical_cases"])
print(f"\nInitial trust indicator: {presentation['trust_score']:.1%}")
print("\nDispatcher interaction options:")
for option in presentation["interaction_options"]:
    print(f"- {option['label']}: {option['description']}")

In [ ]:
print("\n" + "=" * 60)
print("DISPATCHER DECISION AND FEEDBACK")
print("=" * 60)

# This is a demonstration input, not evidence that the recommendation is objectively correct.
human_decision = {
    "action": "accept",
    "reasoning": "The maintenance and fuel penalties are relevant; verify the truck condition before dispatch.",
    "confidence_in_ai": 0.85,
}
print(f"Action: {human_decision['action']}")
print(f"Dispatcher rationale: {human_decision['reasoning']}")

decision_result = symbiotic_interface.process_human_decision(session_id, human_decision)
if "error" in decision_result:
    raise RuntimeError(decision_result["error"])
adaptation_event = adaptation_engine.adapt_to_feedback(
    {"action": human_decision["action"]}, recommendation
)

print(f"\nFeedback recorded: {decision_result['feedback_acknowledged']}")
print(f"Updated trust indicator: {decision_result['updated_trust']:.1%}")
print(f"Trust change in this interaction: {decision_result['learning_outcomes']['trust_change']:+.1%}")
print(f"Dispatch rule retrained: {decision_result['learning_outcomes']['dispatch_rule_retrained']}")
print(f"Adaptation event: {adaptation_event}")
print(f"Confidence-feedback comparison score: {ai_system.confidence_calibrators.get_calibration_score():.1%}")
print("The comparison score summarizes agreement with one supplied rating; it is not a validation study.")

In [ ]:
print("\n" + "=" * 60)
print("INTERACTION LOOP DEMONSTRATION")
print("=" * 60)

scenarios = [
    {
        "name": "High-priority request",
        "action": "accept",
        "description": "Review a context containing another high-priority container.",
    },
    {
        "name": "Resource constraint",
        "action": "modify",
        "description": "Review a context in which all trucks become available later.",
    },
    {
        "name": "Emergency request",
        "action": "accept",
        "description": "Review an additional urgent container and its deadline feasibility.",
    },
]
learning_results = []

for index, scenario in enumerate(scenarios, start=1):
    scenario_context = copy.deepcopy(operational_context)
    if scenario["name"] == "High-priority request":
        scenario_context["containers"].append(
            {"id": f"C-HIGH-{index}", "origin": (150, 80), "destination": (750, 280), "priority": 18, "deadline": 12.0}
        )
    elif scenario["name"] == "Resource constraint":
        for truck in scenario_context["trucks"]:
            truck["available_time"] = 10.0
    else:
        scenario_context["containers"].append(
            {"id": f"C-EMERGENCY-{index}", "origin": (300, 150), "destination": (500, 200), "priority": 25, "deadline": 5.0, "contents": "emergency medical supplies"}
        )

    scenario_session = symbiotic_interface.start_collaboration_session(
        human_dispatcher, scenario_context
    )
    scenario_presentation = symbiotic_interface.present_recommendation(scenario_session)
    scenario_recommendation = scenario_presentation["recommendation"]
    before = symbiotic_interface.trust_model.get_current_trust(human_dispatcher.dispatcher_id)
    decision = {
        "action": scenario["action"],
        "reasoning": scenario["description"],
        "modifications": ["Dispatcher records a proposed change for review."]
        if scenario["action"] == "modify" else [],
    }
    result = symbiotic_interface.process_human_decision(scenario_session, decision)
    adaptation_engine.adapt_to_feedback({"action": scenario["action"]}, scenario_recommendation)
    after = result["updated_trust"]
    learning_results.append(
        {
            "scenario": scenario["name"],
            "action": scenario["action"],
            "trust_before": before,
            "trust_after": after,
            "confidence": scenario_recommendation.confidence_score,
            "assignments": len(scenario_recommendation.recommended_action["assignments"]),
        }
    )
    print(f"{index}. {scenario['name']}: action={scenario['action']}, trust {before:.1%} → {after:.1%}")
    print(f"   Recommendations produced: {len(scenario_recommendation.recommended_action['assignments'])}")

print("\nInterpretation: scenarios exercise the interaction and logging pathway.")
print("They do not measure operational accuracy or train the dispatch heuristic.")

In [ ]:
print("\n" + "=" * 60)
print("OBSERVED INTERACTION METRICS")
print("=" * 60)

metrics = symbiotic_interface.collaboration_metrics
print(f"Decisions recorded: {metrics['decisions_made']}")
print(f"Accepted recommendations: {metrics['ai_acceptances']}")
print(f"Human modifications or rejections: {metrics['human_overrides']}")
if metrics["decisions_made"]:
    print(f"Acceptance share: {metrics['ai_acceptances'] / metrics['decisions_made']:.1%}")
    print(f"Override share: {metrics['human_overrides'] / metrics['decisions_made']:.1%}")

# These panels summarize only the interactions generated in this notebook.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
scenario_labels = [item["scenario"] for item in learning_results]
trust_before = [item["trust_before"] for item in learning_results]
trust_after = [item["trust_after"] for item in learning_results]
positions = np.arange(len(scenario_labels))

axes[0, 0].plot(positions, trust_before, "o-", label="Trust before", color="steelblue")
axes[0, 0].plot(positions, trust_after, "s-", label="Trust after", color="darkorange")
axes[0, 0].set_title("Illustrative trust updates")
axes[0, 0].set_xlabel("Scenario")
axes[0, 0].set_ylabel("Trust indicator")
axes[0, 0].set_ylim(0, 1)
axes[0, 0].set_xticks(positions, [f"S{i + 1}" for i in positions])
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

counts = defaultdict(int)
for item in learning_results:
    counts[item["action"]] += 1
action_names = list(counts)
action_counts = [counts[name] for name in action_names]
axes[0, 1].bar(action_names, action_counts, color="slateblue")
axes[0, 1].set_title("Recorded actions in demonstration scenarios")
axes[0, 1].set_xlabel("Dispatcher action")
axes[0, 1].set_ylabel("Count")
axes[0, 1].grid(axis="y", alpha=0.3)

axes[1, 0].bar(scenario_labels, [item["assignments"] for item in learning_results], color="seagreen")
axes[1, 0].set_title("Per-container recommendations produced")
axes[1, 0].set_xlabel("Scenario")
axes[1, 0].set_ylabel("Recommendation count")
axes[1, 0].tick_params(axis="x", rotation=20)
axes[1, 0].grid(axis="y", alpha=0.3)

axes[1, 1].scatter(
    [item["confidence"] for item in learning_results],
    [item["trust_after"] for item in learning_results],
    color="purple",
)
axes[1, 1].set_title("Heuristic confidence indicator and trust indicator")
axes[1, 1].set_xlabel("Heuristic confidence indicator")
axes[1, 1].set_ylabel("Trust indicator after interaction")
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(alpha=0.3)

fig.suptitle("Demonstration metrics (not an operational performance evaluation)")
fig.tight_layout()
plt.show()
print("Plots summarize generated demonstration records; they do not establish causal effects or model quality.")

In [ ]:
print("\n" + "=" * 70)
print("HUMAN–AI DISPATCH DEMONSTRATION: FINAL REVIEW")
print("=" * 70)

final_trust = symbiotic_interface.trust_model.get_current_trust(human_dispatcher.dispatcher_id)
feedback_count = len(ai_system.feedback_processor.feedback_history)
print(f"Sessions created: {len(symbiotic_interface.active_sessions)}")
print(f"Feedback events recorded: {feedback_count}")
print(f"Current illustrative trust indicator: {final_trust:.1%}")
print(f"Confidence-feedback observations: {len(ai_system.confidence_calibrators.calibration_history)}")

print("\nWhat this implementation demonstrates:")
print("- Per-container deadline feasibility and a transparent utility score.")
print("- A recommendation trace, factor weights, and score-based alternative-truck scenarios.")
print("- Human acceptance, modification, or rejection recorded as feedback.")
print("- A bounded trust indicator updated by the recorded dispatcher action.")

print("\nWhat the demonstration does not establish:")
print("- Global optimality, simultaneous truck feasibility, or operational safety.")
print("- Empirical accuracy gains, reduced training time, or increased flexibility.")
print("- Model learning: the dispatch heuristic remains unchanged after feedback.")
print("- Calibrated confidence: the confidence value is a hand-designed heuristic indicator.")

assert all(0.0 <= item.confidence_score <= 1.0 for item in ai_system.recommendation_history)
assert 0.0 <= final_trust <= 1.0
assert feedback_count == metrics["decisions_made"]
print("\nConsistency checks passed for recorded confidence, trust, and feedback counts.")

## Design rationale, benefits, and limitations

### Why include a human dispatcher?
A dispatcher may know about conditions that are absent from the supplied data, such as a reported vehicle defect, a temporary access restriction, or an operational exception. The interface therefore presents a recommendation as decision support and makes room for human review. This prototype does not encode those unobserved facts automatically; they must be supplied or assessed by the dispatcher.

### Benefits represented in the example
- **Traceability:** the selected truck's feasibility and utility score can be inspected.
- **Human oversight:** acceptance, modification, and rejection are explicit interaction choices.
- **Feedback records:** decisions and optional confidence ratings are stored for later analysis.
- **Counterfactual comparison:** when another truck is feasible, its score is shown if the selected truck is unavailable.

These are software capabilities demonstrated by the code, not evidence of improved outcomes.

### Limitations to consider
- Independent per-container recommendations can reuse the same truck and are not a complete dispatch plan.
- The score weights and confidence rules are illustrative and have not been validated against operational data.
- Counterfactuals vary truck availability only in the stated comparison; they do not simulate downstream effects.
- Trust updates are fixed increments used to demonstrate bookkeeping. Trust is not a direct measure of safety, correctness, or human reliance.
- Feedback is recorded, but no parameter estimation or model training occurs.

### When this interaction pattern may be useful
A human-in-the-loop interface can be useful when decisions require contextual review, explanations, or accountable approval. Before operational use, the data, constraints, score design, safety procedures, and evaluation measures would need to be validated for the specific terminal.

## Summary

This notebook demonstrates a human–AI interaction pattern around an explainable, per-container truck-dispatch heuristic. It defines deadline feasibility, scores feasible trucks, presents recommendations with their reasoning, and records dispatcher feedback.

The mathematical score and its implementation are intentionally visible so that a reader can trace how priority, travel time, fuel, and maintenance affect a recommendation. The trust indicator and confidence estimate are illustrative bookkeeping values, not calibrated measures. Feedback changes the recorded interaction history and trust indicator; it does not change the dispatch rule.

The central interpretation is therefore limited but useful: an explanation can support human review, while a local recommendation must not be mistaken for a globally feasible or optimal dispatch schedule. No numerical claims about accuracy, training time, or operational flexibility are established by this demonstration.

### Natural extensions
Possible extensions within this interaction pattern include adding validated operational data, recording dispatcher modifications in a structured format, and evaluating recommendations against independently verified outcomes. Such work would require defining the operational constraints and evaluation protocol before drawing performance conclusions.